In [1]:
# esta rotina atualiza o campo comparacao_docs da tabela anterioridades_desc
# portanto primeiro deve ser rodado insert_anterioridades_desc.ipynb para ter esta tabela atualizada
# esta rotina faz a leitura de cada anterioridade listada na tabela anterioridades e obtem a descrição no google patents para fazer resumo
# depois a rotina compara com o pedido em exame, e monsta a string final que é salva no campo comparacao_docs da tabela anterioridades_desc

# pip install mysql-connector-python
import mysql.connector

conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

In [2]:
import pandas as pd 

numero = "PI0808715"
# atualiza no localhost as tres tabelas: carga, anterioridades e anterioridades_desc
comando = f"SELECT * FROM arquivados WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
print(resultado)

[(1011532, '1.1', 'PI0808715', datetime.date(2011, 8, 9), 'dialp', 0, 0), (1475660, '1.3', 'PI0808715', datetime.date(2014, 8, 12), 'dialp', 0, 0), (1501853, '6.6', 'PI0808715', datetime.date(2014, 9, 16), 'dialp', 0, 0), (2788968, '7.1', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 1), (2790019, '15.11', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 0), (2956330, '9.2', 'PI0808715', datetime.date(2017, 9, 19), 'dialp', 0, 2), (3015281, '12.2', 'PI0808715', datetime.date(2017, 12, 19), 'dialp', 0, 0)]


In [3]:
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            data = response.json()
            json_data = json.dumps(data, indent=4)
            return(json_data)
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")

In [ ]:
# conecte na VPN
url = 'https://siscap.inpi.gov.br/adm/pareceres/dicel/1120120181571338559.txt' # 1 doc
numero='112012018157'
codigo = '1338559'
divisao = 'dicel'
url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
print(url)
texto_relatorio = conectar_siscap(url,return_json=False)
print(texto_relatorio)
caminho_do_arquivo=f"pareceres/{divisao}/{numero}{codigo}.txt"
with open(caminho_do_arquivo, 'w', encoding='utf-8') as arquivo:
    arquivo.write(texto_relatorio)

In [7]:
import pandas as pd
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where comparacao_docs='' and numero in (select numero from carga) limit 2;"
comando = f"select * from anterioridades_desc where comparacao_docs='' and numero in (select numero from carga where examinador='abrantes') and modelo='gpt-5-nano';"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista1 = lista
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')

#lista = [x for x in lista if x != "numero"]
#json_data = {"patents": [{"numero": item} for item in lista]}
#print(json_data)
#lista = ['numero','112012018157']
print(lista)

['numero', '122017012058', '122020017521', '112015016028', '112015028917', '102015011582', 'PI1103928', '112015005153', '112021023943', '112016018024', '122020017517', '112015028880']


In [11]:
# ******************************************UPDATE ANTERIORIDADES_DESC CAMPO COMPARACAO_DOCS gpt-5-nano
# certifique-se de rodar a rotina acima conectar_siscap e de estar a VPN ligada
# faça cópia da tabela anterioridades no hostgator

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
import re
from bs4 import BeautifulSoup
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from datetime import date

hoje = date.today()

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=openai_api_key, max_completion_tokens=4096,temperature=1)
out_parser = StrOutputParser()
chain = llm | out_parser

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
#json_data = {"patents": [{"numero": item} for item in lista]}
#json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

data["patents"] = lista
#data["patents"] = ['numero','112012018157'] # lista de numeros especificos

with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        numero = data["patents"][i]
        #numero = '102016016661'
        print(f"Lendo o resumo_cepit de {numero}")
        resumo_cepit = ''
        # url =  http://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM titulo where numero='PI0800882'"
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM titulo where numero='{numero}'" + '"'
        url = f"http://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        json_data = conectar_siscap(url,return_json=True)
        data2 = json.loads(json_data)
        for i in range(0, len(data2["patents"])):
            if data2.get("patents"):
                 resumo_cepit = data2["patents"][0].get("resumo_cepit", "")
        print(f"{numero} : {resumo_cepit}")
        
        if resumo_cepit == '': # 122020019318
            url = f"https://patents.google.com/patent/BR{numero}A2/pt?oq=BR{numero}"
            html = urlopen(url)
            bs = BeautifulSoup(html.read(),'html.parser')
            #print(bs.title)
            nameList = bs.findAll("div", {"class":"abstract"})
            for name in nameList:
                resumo_cepit = name.getText()

            if resumo_cepit == '':
                url = f"https://patents.google.com/patent/BR{numero}B1/pt?oq=BR{numero}"
                html = urlopen(url)
                bs = BeautifulSoup(html.read(),'html.parser')
                #print(bs.title)
                nameList = bs.findAll("div", {"class":"abstract"})
                for name in nameList:
                    resumo_cepit = name.getText()
                    
        if resumo_cepit:
            query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM anterioridades where numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
            print(url)
            try:
                output = ''
                json_data = conectar_siscap(url,return_json=True)
                data1 = json.loads(json_data)
                for i in range(0, len(data1["patents"])):
                    codigo = data1["patents"][i]["codigo"]
                    doc = data1["patents"][i]["doc"]
                    doc = re.sub(r'[A-Z]\d$', '', doc)
    
                    print(f"Lendo relatório do estado da técnica: [{codigo}[ [{doc}]")
                    url = f"https://patents.google.com/patent/{doc}A1/en?oq={doc}"
                    print(url)
                    tentar_novamente = False
                    try:
                        html = urlopen(url)
                        tentar_novamente = False
                    except HTTPError as e:
                        tentar_novamente = True
                    except URLError as e:
                        tentar_novamente = True
                    except Exception as e:
                        tentar_novamente = True
                    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A2/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}B1/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
                
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}B2/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A8/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}Y1/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True

                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}U/en?oq={novo_doc}"
                        print(url)
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True

                    if tentar_novamente:
                        novo_doc = doc[:6] + "0" + doc[6:]
                        url = f"https://patents.google.com/patent/{novo_doc}A1/en?oq={novo_doc}"
                        print(f"url={url}")
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        url = f"https://patents.google.com/patent/{novo_doc}A2/en?oq={novo_doc}"
                        print(f"url={url}")
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        url = f"https://patents.google.com/patent/{novo_doc}B1/en?oq={novo_doc}"
                        print(f"url={url}")
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    print(url)
                    bs = BeautifulSoup(html.read(),'html.parser')
                    #print(bs.title)
                    #nameList = bs.findAll("div", {"class":"abstract"})
                    #resumo_D1 = ''
                    #for name in nameList:
                    #  resumo_D1 = name.getText()
                        
                    texto = ''
                    nameList = bs.findAll("section", {"itemprop":"description"})
                    for name in nameList:
                        texto = name.getText()
                    #print(f"{codigo}: {texto}")
                    print(f"Tamanho texto {codigo}: {len(texto)}")
                    if len(texto) == 0: # se não tiver relatório tente ler o resumo
                        nameList = bs.findAll("div", {"class":"abstract"})
                        for name in nameList:
                            texto = name.getText()
                    print(f"Tamanho texto resumo {codigo}: {len(texto)}")

                    if texto:
                        #texto = texto[:2048]
                        query = f"resuma o documento {codigo} em no máximo quatro parágrafos {codigo}: {texto}"
                        resposta = chain.invoke(query)
                        resposta = resposta.replace(';', ',')
                        resposta = resposta.replace(':', ' ')
                        resposta = resposta.replace('"', '')
                        resposta = resposta.replace("'", '')
                        resposta = resposta.replace("\n", " ")
                        output = output + f"Resumo {codigo}: {doc} {resposta}"
        
                        query = f"faça uma comparação técnica objetiva em no máximo quatro parágrafos, entre a invenção descrita em: ###{resumo_cepit}### com o documento {codigo} apontando as diferenças técnicas: {texto}"
                        #print(query)
                        resposta = chain.invoke(query)
                        resposta = resposta.replace(';', ',')
                        resposta = resposta.replace(':', ' ')
                        resposta = resposta.replace('"', '')
                        resposta = resposta.replace("'", '')
                        resposta = resposta.replace("\n", " ")
                        output = output + f", Comparacao: {resposta}. "
                    
                #output = format_as_single_paragraph(output)
                if output:
                    output = json.dumps(output,ensure_ascii=False, indent=4)
                    sql_resumo = f"UPDATE anterioridades_desc set data='{hoje}', comparacao_docs='{output}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
                    print(sql_resumo)
                    f.write(sql_resumo + "\n")
        
                if i == 2:
                    break
                    
            except Exception as e:
                print(f"Não achei anterioridades para {numero} {e}")

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\IPython\core\interactiveshell.py:3517: UserWarning: WARNING! max_completion_tokens is not default parameter.
                max_completion_tokens was transferred to model_kwargs.
                Please confirm that max_completion_tokens is what you intended.
  if await self.run_code(code, result, async_=asy):
D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo o resumo_cepit de 122017012058
122017012058 : Exemplos particulares descritos no presente documento fornecem um dispositivo eletrônico, tal como um computador do tipo notebook ou laptop, o qual inclui uma placa de circuito acoplada a uma pluralidade de componentes eletrônicos (a qual inclui qualquer tipo de hardware, elementos, conjunto de circuitos, etc.). O dispositivo eletrônico também pode incluir uma montagem de conector que é posicionada dentro de pelo menos uma porção de uma reentrância do dispositivo eletrônico, em que a montagem de conector inclui: uma primeira montagem que é destinada a receber um conector, e uma segunda montagem que é destinada a receber um módulo de identificação que é destinado a fornecer uma associação entre um usuário e o dispositivo eletrônico
https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM anterioridades where numero='122017012058'"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Não achei anterioridades para 122017012058 Expecting value: line 1 column 14 (char 13)
Lendo o resumo_cepit de 122020017521
122020017521 : None
Lendo o resumo_cepit de 112015016028
112015016028 : A presente invenção refere-se a sistemas e métodos de computador para identificar uma transação de cartão de pagamento potencialmente fraudulenta em progresso, e mitigar as perdas que surgem do completamento de uma transação de cartão de pagamento fraudulenta. O sistema de computador está programado para receber uma mensagem de solicitação de autorização para a autorização de uma transação iniciada com cartão de pagamento, quando a transação for iniciada utilizando um cartão de pagamento que inclui um primeiro dispositivo de segurança operável para transações iniciadas dentro de uma região geográfica predefinida e um segundo dispositivo de segurança operável para transações iniciadas tanto dentro da região geográfica predefinida quanto fora da região geográfica predefinida. Quando a transação 

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Lendo relatório do estado da técnica: [D1[ [US20090312075]
https://patents.google.com/patent/US20090312075A1/en?oq=US20090312075
https://patents.google.com/patent/US20090312075A1/en?oq=US20090312075
Tamanho texto D1: 14331
Tamanho texto resumo D1: 14331
Lendo relatório do estado da técnica: [D2[ [US20170056722]
https://patents.google.com/patent/US20170056722A1/en?oq=US20170056722
https://patents.google.com/patent/US20170056722A1/en?oq=US20170056722
Tamanho texto D2: 115632
Tamanho texto resumo D2: 115632
UPDATE anterioridades_desc set data='2026-04-08', comparacao_docs='"Resumo D1: US20090312075 Resumo em até quatro parágrafos do documento D1   Este documento aborda dispositivos móveis, especialmente telefones com tampa (flip/clam) que podem abrir ou fechar o espaço entre a tampa e o corpo do aparelho. Tradicionalmente, a detecção do estado aberto/fechado dependia de sensores magnéticos dedicados, o que ocupa espaço adicional. A invenção propõe uma técnica alternativa para determinar e

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo relatório do estado da técnica: [D1[ [US2010278505]
https://patents.google.com/patent/US2010278505A1/en?oq=US2010278505
https://patents.google.com/patent/US2010278505A2/en?oq=US2010278505
https://patents.google.com/patent/US2010278505B1/en?oq=US2010278505
https://patents.google.com/patent/US2010278505B2/en?oq=US2010278505
https://patents.google.com/patent/US2010278505A/en?oq=US2010278505
https://patents.google.com/patent/US2010278505A8/en?oq=US2010278505
https://patents.google.com/patent/US2010278505Y1/en?oq=US2010278505
https://patents.google.com/patent/US2010278505U/en?oq=US2010278505
url=https://patents.google.com/patent/US20100278505A1/en?oq=US20100278505
https://patents.google.com/patent/US20100278505A1/en?oq=US20100278505
Tamanho texto D1: 13037
Tamanho texto resumo D1: 13037
UPDATE anterioridades_desc set data='2026-04-08', comparacao_docs='"Resumo D1: US2010278505 O D1 descreve um dispositivo eletrônico, como uma câmera digital ou TV, equipado com um sistema de edição de 

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo relatório do estado da técnica: [D1[ [US2010082458]
https://patents.google.com/patent/US2010082458A1/en?oq=US2010082458
https://patents.google.com/patent/US2010082458A2/en?oq=US2010082458
https://patents.google.com/patent/US2010082458B1/en?oq=US2010082458
https://patents.google.com/patent/US2010082458B2/en?oq=US2010082458
https://patents.google.com/patent/US2010082458A/en?oq=US2010082458
https://patents.google.com/patent/US2010082458A8/en?oq=US2010082458
https://patents.google.com/patent/US2010082458Y1/en?oq=US2010082458
https://patents.google.com/patent/US2010082458U/en?oq=US2010082458
url=https://patents.google.com/patent/US20100082458A1/en?oq=US20100082458
https://patents.google.com/patent/US20100082458A1/en?oq=US20100082458
Tamanho texto D1: 25023
Tamanho texto resumo D1: 25023
UPDATE anterioridades_desc set data='2026-04-08', comparacao_docs='"Resumo D1: US2010082458 O documento descreve um sistema de gestão adaptativa de níveis críticos de estoque para itens médicos, visand

In [9]:
import pandas as pd
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from anterioridades_desc where comparacao_docs='' and numero in (select numero from carga) limit 2;"
comando = f"select * from anterioridades_desc where comparacao_docs='' and numero in (select numero from carga where examinador in ('abrantes','milavsl','scaplan','srosa','cidade','eloliveira','apedrosa','rosanab','jsoares','alciclea','szandona','douglasm','cujikawa')) and modelo='qwen/qwen3-32b';"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
lista1 = lista
lista = df.iloc[:, 1].tolist()
lista.insert(0, 'numero')
print(lista)

['numero', '112015029260', '102015025507', '122020021404', '102014011129', '102015006751', '112012007444', '112012022998', '112012025948', '112013019993', '112013026489', '112014000630', '112014031962', '112015000416', '112015010392', '112015032570', '122021005431', '202013019250', '202013024035', 'MU9100240', 'PI0903718', 'PI0906951', 'PI0908768', 'PI0915852', 'PI0919409', 'PI1004831', 'PI1005697', 'PI1010927', '112015030352', '112016013519', '102012021502', '102013017278', '102013019765', '102013025651', '102014021906', '102015007934', '102016027868', '112012029640', '112013019699', '112013022994', '112013025238', '112013029300', '112013029498', '112014001086', '112014009986', '112014011387', '112014014731', '112016003414', '112016004649', '112016008784', '112016010521', '112016018219', '122020016644', '122020022012', '122020022821', '122021006465', '122022020636', '202014013394', '202015030495', 'PI1104976', '102012004079', '102012021084', '102012023815', '102012030377', '1020130142

In [10]:
# ******************************************UPDATE ANTERIORIDADES_DESC CAMPO COMPARACAO_DOCS qwen/qwen3-32b
# certifique-se de rodar a rotina acima conectar_siscap e de estar a VPN ligada
# faça cópia da tabela anterioridades no hostgator
#costuma estourar prompt Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization 
# `org_01j476gt68ewe92hjjydwje2td` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6465, 
# please reduce your message size and try again. 

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from bs4 import BeautifulSoup
from urllib.request import urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from datetime import date
import re
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()
    
load_dotenv(dotenv_path='.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="qwen/qwen3-32b")
out_parser = StrOutputParser()
chain = llm | out_parser

hoje = date.today()

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
#json_data = {"patents": [{"numero": item} for item in lista]}
#json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

data["patents"] = lista
#data["patents"] = ['numero','112012018157'] # lista de numeros especificos

with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        numero = data["patents"][i]
        #numero = '102016016661'
        print(f"Lendo o resumo_cepit de {numero}")
        resumo_cepit = ''
        # url =  http://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM titulo where numero='PI0800882'"
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM titulo where numero='{numero}'" + '"'
        url = f"http://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        json_data = conectar_siscap(url,return_json=True)
        data2 = json.loads(json_data)
        for i in range(0, len(data2["patents"])):
            if data2.get("patents"):
                 resumo_cepit = data2["patents"][0].get("resumo_cepit", "")
        print(f"{numero} : {resumo_cepit}")
        
        if resumo_cepit == '': # 122020019318
            url = f"https://patents.google.com/patent/BR{numero}A2/pt?oq=BR{numero}"
            html = urlopen(url)
            bs = BeautifulSoup(html.read(),'html.parser')
            #print(bs.title)
            nameList = bs.findAll("div", {"class":"abstract"})
            for name in nameList:
                resumo_cepit = name.getText()

            if resumo_cepit == '':
                url = f"https://patents.google.com/patent/BR{numero}B1/pt?oq=BR{numero}"
                html = urlopen(url)
                bs = BeautifulSoup(html.read(),'html.parser')
                #print(bs.title)
                nameList = bs.findAll("div", {"class":"abstract"})
                for name in nameList:
                    resumo_cepit = name.getText()
                    
        if resumo_cepit:
            query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM anterioridades where numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
            print(url)
            try:
                output = ''
                json_data = conectar_siscap(url,return_json=True)
                data1 = json.loads(json_data)
                for i in range(0, len(data1["patents"])):
                    codigo = data1["patents"][i]["codigo"]
                    doc = data1["patents"][i]["doc"]
    
                    print(f"Lendo relatório do estado da técnica: {codigo} {doc}")
                    url = f"https://patents.google.com/patent/{doc}A1/en?oq={doc}"
                    tentar_novamente = False
                    try:
                        html = urlopen(url)
                        tentar_novamente = False
                    except HTTPError as e:
                        tentar_novamente = True
                    except URLError as e:
                        tentar_novamente = True
                    except Exception as e:
                        tentar_novamente = True
                    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A2/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}B1/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
                
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}B2/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}A8/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}Y1/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True

                    if tentar_novamente:
                        novo_doc = doc
                        url = f"https://patents.google.com/patent/{novo_doc}U/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True

                    if tentar_novamente:
                        novo_doc = doc[:6] + "0" + doc[6:]
                        url = f"https://patents.google.com/patent/{novo_doc}A1/en?oq={novo_doc}"
                        print(f"url={url}")
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        url = f"https://patents.google.com/patent/{novo_doc}A2/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    if tentar_novamente:
                        url = f"https://patents.google.com/patent/{novo_doc}B1/en?oq={novo_doc}"
                        tentar_novamente = False
                        try:
                            html = urlopen(url)
                            tentar_novamente = False
                        except HTTPError as e:
                            tentar_novamente = True
                        except URLError as e:
                            tentar_novamente = True
                        except Exception as e:
                            tentar_novamente = True
    
                    print(url)
                    bs = BeautifulSoup(html.read(),'html.parser')
                    #print(bs.title)
                    #nameList = bs.findAll("div", {"class":"abstract"})
                    #resumo_D1 = ''
                    #for name in nameList:
                    #  resumo_D1 = name.getText()
                        
                    texto = ''
                    nameList = bs.findAll("section", {"itemprop":"description"})
                    for name in nameList:
                        texto = name.getText()
                    #print(f"{codigo}: {texto}")
                    print(f"Tamanho texto {codigo}: {len(texto)}")
                    if len(texto) == 0: # se não tiver relatório tente ler o resumo
                        nameList = bs.findAll("div", {"class":"abstract"})
                        for name in nameList:
                            texto = name.getText()
                    print(f"Tamanho texto resumo {codigo}: {len(texto)}")

                    if texto:
                        #texto = texto[:2048]
                        query = f"resuma o documento {codigo} em no máximo quatro parágrafos {codigo}: {texto}"
                        messages=[{"role":"user", "content": query}]
                        response = llm.invoke(messages)
                        resposta = clean_answer(response.content)
                        time.sleep(3) 
                        
                        resposta = resposta.replace(';', ',')
                        resposta = resposta.replace(':', ' ')
                        resposta = resposta.replace('"', '')
                        resposta = resposta.replace("'", '')
                        resposta = resposta.replace("\n", " ")
                        output = output + f"Resumo {codigo}: {doc} {resposta}"
        
                        query = f"faça uma comparação técnica objetiva em no máximo quatro parágrafos, entre a invenção descrita em: ###{resumo_cepit}### com o documento {codigo} apontando as diferenças técnicas: {texto}"
                        #print(query)
                        messages=[{"role":"user", "content": query}]
                        response = llm.invoke(messages)
                        resposta = clean_answer(response.content)
                        time.sleep(3) 

                        resposta = resposta.replace(';', ',')
                        resposta = resposta.replace(':', ' ')
                        resposta = resposta.replace('"', '')
                        resposta = resposta.replace("'", '')
                        resposta = resposta.replace("\n", " ")
                        output = output + f", Comparacao: {resposta}. "
                    
                #output = format_as_single_paragraph(output)
                if output:
                    output = json.dumps(output,ensure_ascii=False, indent=4)
                    sql_resumo = f"UPDATE anterioridades_desc set data='{hoje}', comparacao_docs='{output}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
                    print(sql_resumo)
                    f.write(sql_resumo + "\n")
        
                if i == 2:
                    break
                    
            except Exception as e:
                print(f"Não achei anterioridades para {numero} {e}")

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo o resumo_cepit de 112015029260
112015029260 : A presente invenção está relacionada a novos derivados de macrólidos, em particular, novos derivados de tilosina da fórmula (IIa), (IIa),uma composição farmacêutica ou veterinária que consiste nos derivados, um método para sua comparação, um método para tratamento e/ou prevenção de infecções bacterianas em um animal, no qual o método consiste na administração dos derivados ou composição e um uso de derivados para a fabricação de medicamentos para tratar e/ou prevenir infecções bacterianas em um animal
https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM anterioridades where numero='112015029260'"


D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo relatório do estado da técnica: D1 WO9402496
url=https://patents.google.com/patent/WO94020496A1/en?oq=WO94020496
https://patents.google.com/patent/WO94020496B1/en?oq=WO94020496
Tamanho texto D1: 0
Tamanho texto resumo D1: 0
Lendo relatório do estado da técnica: D3 WO2008012343
https://patents.google.com/patent/WO2008012343A2/en?oq=WO2008012343
Tamanho texto D3: 97083
Tamanho texto resumo D3: 97083
Não achei anterioridades para 112015029260 Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01j476gt68ewe92hjjydwje2td` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 25099, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Lendo o resumo_cepit de 102015025507
102015025507 : A presente invenção refere-se a um conjunto de equipamentos que possibilita acumular mercador

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo relatório do estado da técnica: D1 US2005063801
url=https://patents.google.com/patent/US20050063801A1/en?oq=US20050063801
https://patents.google.com/patent/US20050063801A1/en?oq=US20050063801
Tamanho texto D1: 28934
Tamanho texto resumo D1: 28934
Não achei anterioridades para 102015025507 Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01j476gt68ewe92hjjydwje2td` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6465, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Lendo o resumo_cepit de 122020021404
122020021404 : None
Lendo o resumo_cepit de 102014011129
102014011129 : A presente invenção refere-se a um Sistema para realizar medição da temperatura em cabos condutores subterrâneos, com sensores pontualmente distribuídos estrategicamente ao longo da linha par

D:\Users\abrantes\InstallAnaconda\envs\python\lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cientistaspatentes.com.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lendo relatório do estado da técnica: D1 CN101709981
https://patents.google.com/patent/CN101709981A/en?oq=CN101709981
Tamanho texto D1: 13787
Tamanho texto resumo D1: 13787


KeyboardInterrupt: 